BLOQUE 1: Carga y Limpieza "Estilo Crítico"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Cargar datos
df = pd.read_csv("archivo.csv")

# 2. EL PASO CLAVE: Detectar y limpiar el "?" 
# (Muy común en tus notebooks de 'Adult')
df.replace('?', np.nan, inplace=True)

# 3. Ver nulos reales ahora
print(df.isnull().sum())
df.dropna(inplace=True) # El profesor suele preferir borrar si hay suficientes datos

# 4. Separar X e y
target = 'nombre_columna_objetivo'
X = df.drop(target, axis=1)
y = df[target]

BLOQUE 2: Preparación

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# 1. Identificar tipos de columnas
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

# 2. Split con STRATIFY (Para mantener proporciones de clases)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Preprocesador (ColumnTransformer) - Muy usado en EstudioDatos_03
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

# 4. Ajustar SOLO con train y transformar ambos
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

BLOQUE 3: Entrenamiento de Modelos (Los 5 Fantásticos)

A. Regresión Logística (El modelo base)

In [ ]:
from sklearn.linear_model import LogisticRegression
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_prep, y_train)

B. K-Nearest Neighbors (KNN)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=5) # El profesor suele preguntar por n_neighbors
knn.fit(X_train_prep, y_train)

C. Support Vector Machine (SVM)

In [ ]:
from sklearn.svm import SVC
# Kernel RBF (No lineal, más flexible)
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)
svm_rbf.fit(X_train_prep, y_train)

# Kernel Lineal (Si los datos son simples)
svm_lin = SVC(kernel='linear', C=1.0)
svm_lin.fit(X_train_prep, y_train)


# Grado 2 (curva parabólica) o Grado 3 (curva en S)
svm_poly = SVC(kernel='poly', degree=3, C=1.0, coef0=1) 
svm_poly.fit(X_train_prep, y_train)



D. Árboles y Random Forest (Clasificación o Regresión)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

# 1. Decision Tree (Un solo árbol, muy interpretable)
dt_clf = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_clf.fit(X_train_prep, y_train)

# 2. Random Forest (Muchos árboles, más robusto)
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_clf.fit(X_train_prep, y_train)

# Evaluación: Usamos Accuracy, Recall, Matriz de Confusión

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

# 1. Decision Tree Regressor
dt_reg = DecisionTreeRegressor(max_depth=5, random_state=42)
dt_reg.fit(X_train_prep, y_train)

# 2. Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train_prep, y_train)

# Evaluación: Usamos MSE (Error) y R2 (Precisión)
from sklearn.metrics import mean_squared_error, r2_score
y_pred = rf_reg.predict(X_test_prep)
print(f"MSE: {mean_squared_error(y_test, y_pred)}")
print(f"R2: {r2_score(y_test, y_pred)}")

BLOQUE 4: Evaluación y Métricas (Parte Práctica)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score

def evaluar(modelo, name):
    y_pred = modelo.predict(X_test_prep)
    print(f"--- {name} ---")
    print(classification_report(y_test, y_pred))
    
    # Matriz de Confusión visual (Estilo Breast Cancer notebook)
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap='Blues')
    plt.title(f"Matriz {name}")
    plt.show()

evaluar(svm_rbf, "SVM RBF")

In [ ]:
# Solo para modelos que tengan predict_proba (Logística, SVM con probability=True, RF)
y_probs = svm_rbf.predict_proba(X_test_prep)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_probs)
auc = roc_auc_score(y_test, y_probs)

plt.plot(fpr, tpr, label=f"AUC = {auc:.2f}")
plt.plot([0, 1], [0, 1], 'k--')
plt.legend()
plt.show()

5️⃣ BLOQUE 5: Regresión (Si entra en el examen)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
# ... entrenar modelo de regresión (ej. SVR o RandomForestRegressor) ...
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Error Cuadrático Medio: {mse}")
print(f"Precisión R2: {r2}") # Explica cuánta varianza capturas

¿Por qué el Kernel Lineal rinde peor en algunos casos?

Respuesta: Porque si los datos tienen una estructura compleja (como círculos concéntricos o muchas variables interactuando), una línea recta (hiperplano) no puede separarlos. El RBF permite crear fronteras curvas.

¿Qué es el parámetro Gamma en SVM?

Respuesta: Define la "distancia de influencia" de un solo punto. Un Gamma alto hace que el modelo se fije solo en puntos muy cercanos (riesgo de overfitting). Un Gamma bajo hace que el modelo sea más suave.

¿Qué es el parámetro C?

Respuesta: Es el equilibrio entre el error y el margen. Un C alto penaliza mucho los errores (ajusta más al entrenamiento), un C bajo permite algunos errores a cambio de un margen más ancho (más generalizable).

¿Accuracy o AUC en medicina?

Respuesta: En medicina (como el dataset de Cáncer o Diabetes) es mejor el AUC o el Recall, porque nos importa mucho no tener "Falsos Negativos" (decirle a alguien enfermo que está sano).

¿KNN con muchos datos?

Respuesta: No es recomendable. KNN tiene que calcular la distancia de cada punto nuevo contra todos los puntos del dataset. Si el dataset crece 100 veces, el modelo se vuelve extremadamente lento.

¿Por qué escalar?

Respuesta: Modelos como KNN y SVM usan distancias. Si la "Edad" va de 0-100 y el "Salario" de 0-100.000, el salario se comerá a la edad en el cálculo y el modelo ignorará la edad.

El parámetro degree (Grado):

Es la potencia del polinomio. Un grado 1 es equivalente a un kernel lineal.

A mayor grado, más se curva la frontera de decisión para intentar rodear los datos.

Peligro: Un grado muy alto (ej. degree=10) causará un sobreajuste (overfitting) extremo, el modelo será "un esclavo" de los puntos del entrenamiento.

El parámetro coef0 (Término independiente):

Controla cuánto influyen los términos de grado alto frente a los de grado bajo. En tus apuntes suele aparecer como un valor que ayuda a que el modelo no se vuelva loco con grados altos.

¿Cuándo usarlo frente a RBF?

Poli: Es útil cuando sabes que la relación entre variables es de tipo potencia (ej. el riesgo crece con el cuadrado de la velocidad).

RBF: Es el "comodín". En la práctica, RBF suele dar mejores resultados que el Polinómico porque es más flexible y más fácil de ajustar (solo tiene gamma).

¿Qué es n_estimators?

Respuesta: Es el número de árboles que va a crear el bosque. Cuantos más árboles, más estable es el modelo, pero más lento de entrenar. Normalmente 100 es un buen equilibrio.

¿Para qué sirve max_depth?

Respuesta: Controla la profundidad del árbol. Si no se limita, el árbol crecerá hasta memorizar cada dato individual (Overfitting). Limitarlo ayuda a que el modelo generalice mejor para datos nuevos.

¿Por qué Random Forest suele ser mejor que un solo Árbol de Decisión?

Respuesta: Un solo árbol es muy sensible a pequeñas variaciones en los datos (tiene mucha varianza). El Random Forest, al promediar las predicciones de muchos árboles diferentes, reduce ese error y es mucho más preciso.

¿Qué es la "Importancia de las Variables" (Feature Importance)?

Respuesta: Random Forest permite ver qué columnas han sido más útiles para decidir. Esto responde a la pregunta del profesor: "¿Qué variables han resultado más informativas?".

1. Sobre el Preprocesamiento y Limpieza
P: ¿Por qué es fundamental realizar el escalado de variables en modelos como KNN o SVM?

Respuesta: Porque ambos modelos se basan en el cálculo de distancias (como la euclídea). Si una variable tiene un rango de 0 a 1,000,000 (ej. salario) y otra de 0 a 10 (ej. años de educación), la variable de mayor rango dominará completamente el modelo, haciendo que la otra sea irrelevante. El escalado pone a todas en la misma "escala".

P: ¿Por qué el consejo dice "NO escales antes de dividir en train/test"?

Respuesta: Para evitar el data leakage (fuga de datos). Si escalas antes del split, el StandardScaler conocerá la media y desviación del conjunto de test, "contaminando" el entrenamiento con información que, en teoría, el modelo no debería conocer.

P: En el dataset 'Adult', ¿qué problema presentan los valores marcados con "?" y cómo se resuelven?

Respuesta: Son valores ausentes (NaN) que Python no reconoce como tales automáticamente. Primero deben reemplazarse con np.nan y luego decidir si eliminarlos con dropna() o imputarlos (rellenarlos) con la media o la moda.

2. Sobre Support Vector Machines (SVM)
P: ¿Qué diferencia hay entre un Kernel Lineal y un Kernel RBF?

Respuesta: El lineal busca un hiperplano recto para separar las clases (útil si los datos son linealmente separables). El RBF (Radial Basis Function) proyecta los datos a una dimensión superior para encontrar fronteras de decisión curvas y complejas.

P: ¿Qué sucede si el parámetro Gamma en un kernel RBF es muy elevado?

Respuesta: El modelo intentará ajustar la frontera de decisión de forma muy exacta a cada punto del entrenamiento. Esto provoca overfitting (sobreajuste), lo que significa que el modelo memoriza el ruido y fallará con datos nuevos.

P: ¿Para qué sirve el parámetro C?

Respuesta: Es el parámetro de regularización. Un C alto penaliza mucho los errores de clasificación (margen estrecho, riesgo de overfitting). Un C bajo permite algunos errores a cambio de un margen más ancho y un modelo más generalizable.

3. Sobre Árboles y Random Forest
P: ¿Por qué un Random Forest suele ser mejor que un solo Árbol de Decisión?

Respuesta: Un árbol individual tiene mucha "varianza" (es muy inestable y tiende al overfitting). El Random Forest reduce esta varianza al promediar las predicciones de múltiples árboles entrenados con diferentes subconjuntos de datos.

P: ¿Qué indica la métrica Feature Importance en un Random Forest?

Respuesta: Indica cuánto contribuye cada variable a reducir la impureza de los nodos del árbol. Ayuda a identificar qué características (ej. edad, nivel educativo) son las que realmente determinan la predicción del modelo.

4. Sobre Evaluación y Métricas
- P: En un contexto médico (ej. Cáncer de Mama), ¿qué métrica es más importante: Accuracy o Recall?
Respuesta: El Recall (o Sensibilidad). En medicina, el coste de un "Falso Negativo" (decir que alguien está sano cuando está enfermo) es crítico. El Recall nos asegura que estamos detectando el máximo número de casos positivos reales.
- P: ¿Cuándo es preferible usar el AUC (Área bajo la curva ROC) en lugar de la Accuracy?Respuesta: Cuando el dataset está desbalanceado (ej. 95% de personas sanas y 5% enfermas). Un modelo que siempre diga "sano" tendrá un 95% de Accuracy pero será inútil. El AUC evalúa la capacidad del modelo para distinguir entre ambas clases sin importar el umbral.
- P: ¿Qué significa que un modelo de regresión tenga un $R^2$ de 0.90?Respuesta: Significa que el modelo es capaz de explicar el 90% de la variabilidad de los datos observados. Cuanto más cerca de 1, mejor ajustado está el modelo a la realidad.

5. Comparativa Final (La pregunta del millón)
P: Si el dataset fuera 100 veces más grande, ¿qué modelo sería el menos recomendable y por qué?

Respuesta: El KNN. KNN no "aprende" realmente una fórmula; guarda todos los puntos y calcula distancias cada vez que llega un dato nuevo. Con millones de datos, el tiempo de cálculo para cada predicción sería inasumible.

Cuando el profesor te pida conclusiones, usa siempre esta frase mágica:

"Aunque el modelo X tiene un Accuracy similar al modelo Y, elijo el modelo [X] porque presenta un mejor [Recall/AUC], lo que garantiza una mayor robustez ante datos no vistos y minimiza los errores críticos en la clase positiva."